### Imports and global setup

In [1]:
## conda env: stereo_visionn
import os
import cv2
import glob
import numpy as np
from rtmlib import Wholebody, draw_skeleton, draw_bbox
from Util.util import BodyWithFeet, PoseTracker, Body, Custom, pose_to_bbox
import time
import json
from IPython.display import display, clear_output
from functools import partial

with open('./Util/models.json') as f:
    models = json.load(f)

#### Stock setup

In [ ]:
device = "cuda"  # cpu, cuda, mps
backend = "onnxruntime"  # opencv, onnxruntime, openvino
openpose_skeleton = False  # True for openpose-style, False for mmpose-style

body = Body(
    to_openpose=openpose_skeleton, mode="performance", backend=backend, device=device
)

load C:\Users\unger\.cache\rtmlib\hub\checkpoints\yolox_x_8xb8-300e_humanart-a39d44ed.onnx with onnxruntime backend
load C:\Users\unger\.cache\rtmlib\hub\checkpoints\rtmpose-x_simcc-body7_pt-body7_700e-384x288-71d7b7e9_20230629.onnx with onnxruntime backend


#### Custom detector & model

In [ ]:
device = "cuda"  # cpu, cuda, mps
backend = "onnxruntime"  # opencv, onnxruntime, openvino
detector_name = 'YOLOX_nano' # 'YOLOX_l_COCO','YOLOX_nano','YOLOX_tiny','YOLOX_s','YOLOX_m','YOLOX_l','YOLOX_x'
pose_name = 'RTMPose_x' # (26) 'RTMPose_t', 'RTMPose_s', 'RTMPose_m', 'RTMPose_l', 'RTMPose_m2', 'RTMPose_l2', 'RTMPose_x', (133) 'RTMW_l', 'RTMW_x'

custom = Custom(det_class='YOLOX',#'RTMDet',
                det=models['detectors'][detector_name]['path'],
                det_input_size=models['detectors'][detector_name]['input_size'],
                pose_class='RTMPose',
                pose=models['pose_models'][pose_name]['path'],
                pose_input_size=models['pose_models'][pose_name]['input_size'],
                backend=backend,
                device=device) 

# custom = partial(Custom,
#                 det_class='YOLOX',#'RTMDet',
#                 det=models['detectors'][detector_name]['path'],
#                 det_input_size=models['detectors'][detector_name]['input_size'],
#                 pose_class='RTMPose',
#                 pose=models['pose_models'][pose_name]['path'],
#                 pose_input_size=models['pose_models'][pose_name]['input_size'],
#                 backend=backend,
#                 device=device)

# pose_tracker = PoseTracker(custom,
#                            det_frequency=1,
#                            to_openpose=False,
#                            backend=backend, device=device)

load C:\Users\unger\.cache\rtmlib\hub\checkpoints\yolox_nano_8xb8-300e_humanart-40f6f0d0.onnx with onnxruntime backend
load C:\Users\unger\.cache\rtmlib\hub\checkpoints\rtmw-dw-x-l_simcc-cocktail14_270e-384x288_20231122.onnx with onnxruntime backend


### Mapping keypoints

In [ ]:
img = cv2.imread("./Util/tennis.webp", cv2.IMREAD_COLOR)
keypoints, scores = custom(img)

np.save('./keypoints_rtmposex.npy',keypoints)

In [5]:
keypoints.shape

(1, 133, 2)

In [29]:
custom = Custom(det_class='YOLOX',#'RTMDet',
                det=models['detectors'][detector_name]['path'],
                det_input_size=models['detectors'][detector_name]['input_size'],
                pose_class='RTMPose',
                pose="https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-face6_pt-in1k_120e-256x256-72a37400_20230529.zip",
                pose_input_size=[256,256],
                backend=backend,
                device=device)

Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-face6_pt-in1k_120e-256x256-72a37400_20230529.zip" to C:\Users\unger\.cache\rtmlib\hub\checkpoints\rtmpose-m_simcc-face6_pt-in1k_120e-256x256-72a37400_20230529.zip


load C:\Users\unger\.cache\rtmlib\hub\checkpoints\yolox_nano_8xb8-300e_humanart-40f6f0d0.onnx with onnxruntime backend


100%|██████████| 60.3M/60.3M [00:05<00:00, 12.6MB/s]


load C:\Users\unger\.cache\rtmlib\hub\checkpoints\rtmpose-m_simcc-face6_pt-in1k_120e-256x256-72a37400_20230529.onnx with onnxruntime backend


In [55]:
# img = cv2.imread("./Util/tennis.webp", cv2.IMREAD_COLOR)
# img = cv2.imread("./stereo_videos/validation_test/toes.png", cv2.IMREAD_COLOR)
img = cv2.imread('./asdf.png', cv2.IMREAD_COLOR)
# cv2.imshow("Image", img)
keypoints, scores = custom(img)
# keypoints, scores = body(img)

for index, point in enumerate(keypoints[0]):
    # if index<83 and index >= 66:

    cv2.circle(img, (round(point[0]), round(point[1])), 5, (0, 0, 255), 3)
    cv2.putText(
        img,
        f"{index+1}",
        (round(point[0] + 5), round(point[1]) + 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 100, 0),
        3,
    )
        # print(point, index)

cv2.imshow(
    "Image", cv2.resize(img, (round(img.shape[1] * 0.3), round(img.shape[0] * 0.3)))
)
# cv2.imwrite("./Util/Body_marker_locations_133.png", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

### Video analysis

In [ ]:
# input_folder = "F:/sl_validation_data/Mate/walking_B88C87A3-6562-4675-8091-648CE3B7E3DE"
input_folder = './stereo_videos'
input_folder = input_folder.replace('\\','/')
# file_name = "me.mp4"
for g in glob.glob(os.path.join(input_folder,"*.avi")):
    print(g)
    file_name = g.split('\\')[-1].split('.')[0]
    print(file_name)

F:/sl_validation_data/Mate/walking_B88C87A3-6562-4675-8091-648CE3B7E3DE\43804892.avi
43804892
F:/sl_validation_data/Mate/walking_B88C87A3-6562-4675-8091-648CE3B7E3DE\43916681.avi
43916681


In [ ]:
keypoints_over_time = np.array([])

# input_folder = "./stereo_videos/"
# input_folder = "F:\sl_validation_data\Mate\stickfollowing_FEEA6CCA-7265-498F-B066-3BFB164DF478"
# input_folder = input_folder.replace("\\", "/")
file_name = "me.mp4"
for g in glob.glob(os.path.join(input_folder, "*.avi")):
    print(g)
    file_name = g.split("\\")[-1]

    cap = cv2.VideoCapture(os.path.join(input_folder, file_name))

    if cap.isOpened() == False:
        print("Error opening video file")

    num_frames = 0
    start_time = time.time_ns()
    # Read until video is completed
    while cap.isOpened():

        # Capture frame-by-frame
        ret, frame = cap.read()

        if ret == True and frame is not None:
            num_frames += 1
            width = frame.shape[1]
            height = frame.shape[0]

            keypoints, scores = custom(frame)
            # keypoints, scores = body(frame)

            # keypoints_over_time.append(clipped_keypoints)
            np.append(keypoints_over_time, keypoints)

            img_show = draw_skeleton(frame, keypoints, scores, kpt_thr=0.6)

            # img_show = draw_skeleton(frame, keypoints, scores, kpt_thr=0.2)
            boxes = [pose_to_bbox(x) for x in keypoints]
            img_show = draw_bbox(frame, boxes, (0, 0, 255))

            ## check to see in ankle tracking switches when changing direction
            # cv2.circle(img_show,(round(keypoints[0][16][0]),round(keypoints[0][16][1])),5,(0,0,255),3 )

            cv2.imshow("Image", cv2.resize(img_show, (int(width / 2), int(height / 2))))

            # Press Q on keyboard to exit
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        else:
            break
    end_time = time.time_ns()

    time_diff_seconds = (end_time - start_time) / 1000000000
    print(f"playback fps: {round(num_frames/time_diff_seconds,2)}")

    cap.release()
    cv2.destroyAllWindows()

    out_folder_name = input_folder
    file_name = file_name.split(".")[0]

    # print(type(keypoints_over_time))
    # keypoints_over_time = np.array(keypoints_over_time)
    print(keypoints_over_time.shape)

    np.save(
        f"{os.path.join(out_folder_name, file_name)}_{detector_name}_{pose_name}.npy",
        keypoints_over_time,
    )

# keypoints_over_time = np.array(keypoints_over_time)
# print(keypoints_over_time.shape)

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
C:\Users\unger\AppData\Local\Temp\ipykernel_13700\2299286616.py:4: SyntaxWarning: invalid escape sequence '\s'
  input_folder = "F:\sl_validation_data\Mate\stickfollowing_FEEA6CCA-7265-498F-B066-3BFB164DF478"


F:/sl_validation_data/Mate/stickfollowing_FEEA6CCA-7265-498F-B066-3BFB164DF478\43804892.avi
playback fps: 0.84
(0,)
F:/sl_validation_data/Mate/stickfollowing_FEEA6CCA-7265-498F-B066-3BFB164DF478\43916681.avi
playback fps: 0.94
(0,)


In [127]:
cap = cv2.VideoCapture(f'./stereo_videos/validation_test/ken_walking_side_short.mp4')
print(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))



690


In [126]:
input_folder = "./stereo_videos/validation_test"
for g in glob.glob(os.path.join(input_folder, "*ken_walking_side_short.mp4")):
    print(f'processing: {g}')
    file_name = g.split("\\")[-1].split('.')[0]

    # keypoints_over_time = np.empty([0,133,3])
    # keypoints_over_time = np.empty([2,26,3])
    # keypoints_over_time = np.empty([0,2,26,3])
    keypoints_over_time = []


    cap = cv2.VideoCapture(f'./stereo_videos/validation_test/{file_name}.mp4')

    if cap.isOpened() == False:
        print("Error opening video file")

    total_num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    processed_num_frames = 0
    start_time = time.time_ns()


    data_sizes = []
    keypoint_sizes = []

    # Read until video is completed
    while cap.isOpened():

        # Capture frame-by-frame
        ret, frame = cap.read()

        if ret == True and frame is not None:
            processed_num_frames += 1
            width = frame.shape[1]
            height = frame.shape[0]

            # keypoints, scores = custom(frame)
            keypoints, scores = pose_tracker(frame)

            # keypoints_over_time.append(clipped_keypoints)
            # keypoints_over_time = np.append(keypoints_over_time, np.dstack((keypoints, scores)), axis=0)
            # keypoints_over_time = np.concatenate((keypoints_over_time,np.dstack((keypoints, scores))), axis=0)
            keypoints_over_time.append(np.dstack((keypoints, scores)))

            # keypoints_over_time = np.append(keypoints_over_time, np.dstack((keypoints, scores)), axis=2)
            # keypoints_over_time = np.append(keypoints_over_time, np.vstack((keypoints, scores)), axis=0)
            
            # current_data = np.dstack((keypoints, scores))
            # data_sizes.append((processed_num_frames,current_data.shape))
            # keypoint_sizes.append((processed_num_frames, keypoints.shape))
            img_show = draw_skeleton(frame, keypoints, scores, kpt_thr=0.6)

            # img_show = draw_skeleton(frame, keypoints, scores, kpt_thr=0.2)
            # boxes = [pose_to_bbox(x) for x in keypoints]
            # img_show = draw_bbox(frame, boxes, (0, 0, 255))

            ## check to see in ankle tracking switches when changing direction
            # cv2.circle(img_show,(round(keypoints[0][16][0]),round(keypoints[0][16][1])),5,(0,0,255),3 )

            cv2.imshow("Image", cv2.resize(img_show, (int(width / 2), int(height / 2))))

            clear_output(wait=True)
            end_time = time.time_ns()
            elapsed_time = (end_time - start_time) / 1000000000

            if int(elapsed_time % 10) == 0:

                fps=round(processed_num_frames/elapsed_time,2)
                done_ratio = processed_num_frames / total_num_frames
                expected_duration_min = elapsed_time / (done_ratio*60)


                display(f'exp_duration: {round(expected_duration_min,2)} minutes, elapsed: {round(elapsed_time/60,2)} minutes')
                display(f'done: {round(done_ratio*100,2)}%, fps: {fps}')

            # Press Q on keyboard to exit
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        else:
            break

    cap.release()
    cv2.destroyAllWindows()

    print(f'shape of coordinates: {keypoints_over_time.shape}')

    out_file_name = f"./saved_coords/validation_test/{file_name}.npy"
    np.save(
        out_file_name,
        keypoints_over_time,
    )
    print(f"Saved keypoint-coordinates to {out_file_name}")

AttributeError: 'list' object has no attribute 'shape'

In [116]:
from collections import Counter

num_detected_ppl = Counter([len(x) for x in keypoints_over_time])
print(num_detected_ppl)
print(sum(num_detected_ppl.values()))

Counter({1: 143, 2: 69, 3: 1})
213


In [129]:
kp1 = [x[0] for x in keypoints_over_time]
print(len(kp1))
print(kp1[0].shape)
kp1 = np.array(kp1)

out_file_name = f"./saved_coords/validation_test/{file_name}.npy"
np.save(
        out_file_name,
        kp1,
    )
print(f"Saved keypoint-coordinates to {out_file_name}")

690
(26, 3)
Saved keypoint-coordinates to ./saved_coords/validation_test/ken_walking_side_short.npy


In [128]:
processed_num_frames, total_num_frames

(690, 690)

In [88]:
# keypoints_over_time[:8]
# keypoints_over_time = np.empty([0,2,26,3])
print(f'keypoints:  {keypoints.shape}')
print(f'scores:     {scores.shape}')

print(f'stacked:    {np.dstack((keypoints, scores)).shape}')
# print(f'over_time:  {keypoints_over_time.shape}')


keypoints:  (1, 26, 2)
scores:     (1, 26)
stacked:    (1, 26, 3)


#### Save the coords of chosen keypoints

In [ ]:
out_folder_name = "./saved_coords"
# out_folder_name = input_folder
file_name = file_name.split('.')[0]
np.save(
    f"{os.path.join(out_folder_name, file_name)}_{detector_name}_{pose_name}.npy", keypoints_over_time
)